# 벌통 상태 분류 실험 A · 외부 벌통 4-fold와 내부 벌통 3-fold

내부 검증의 단계별 최소 val_loss 에포크 3개의 중앙값을 최종 학습에 사용합니다. 현행 GAP 모델과 60초 WAV 전처리를 사용합니다.

프로젝트 루트에서 커널을 시작하고 모든 셀을 순서대로 실행하세요. 기본값은 전체 학습입니다. 점검만 할 때는 아래 설정에서 `mode`를 `audit`로, 실제 소규모 학습은 `smoke`로 설정하세요. 전체 실행은 매번 새 결과 폴더를 만들고, 점검 결과는 기본적으로 시스템 임시 경로에 저장합니다.

In [ ]:
from pathlib import Path
from dataclasses import asdict, dataclass
from datetime import datetime, timezone, timedelta
from time import perf_counter
import gc
import hashlib
import json
import math
import re
import weakref

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
from keras import layers, models
from sklearn.utils.class_weight import compute_class_weight
from IPython.display import display
import matplotlib.pyplot as plt

EXPERIMENT = 'a'
import tempfile
from uuid import uuid4

RUN_OPTIONS = globals().get('RUN_OPTIONS', {'mode': 'full'})
ROOT = Path(RUN_OPTIONS.get('root', Path.cwd())).resolve()
MODE = RUN_OPTIONS.get('mode', 'full')  # full | audit | smoke
if MODE not in ('full', 'audit', 'smoke'):
    raise ValueError(f'Unknown mode: {MODE}')
SEED = 42
BATCH_SIZE = 32
EPOCHS = 1 if MODE == 'smoke' else 30
SMOKE_PER_CLASS = 2  # Sources per class and split
SILENCE_RMS = 1e-5
MANUAL_EXCLUSIONS = {}
HIVES = ('1', '3', '4', '5')
SCORE_COLUMNS = [f'score_{label}' for label in range(4)]
DEFINITIONS = {
    'a': 'Outer leave-one-hive-out; inner leave-one-hive-out; median best epoch refit',
    'b': 'Fixed source-stratified 70/15/15',
    'c': 'Fixed held source per clock hour with other-hive train source',
}
EXPERIMENTS_DIR = ROOT / 'artifacts/experiments'
if MODE == 'full':
    run_id = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:8]
    OUTPUT = EXPERIMENTS_DIR / f'ex_{EXPERIMENT}' / run_id
    OUTPUT.mkdir(parents=True, exist_ok=False)
else:
    temp_parent = RUN_OPTIONS.get('temporary_parent')
    OUTPUT = Path(tempfile.mkdtemp(prefix=f'bee_ex_{EXPERIMENT}_{MODE}_', dir=temp_parent))
print(f'Experiment {EXPERIMENT.upper()} · {MODE} · {OUTPUT}')

## 1. 로컬 WAV 연결과 원본 배정

원본 관측 시각과 60초 조각의 시작 시각을 구분합니다. 배정은 원본 단위로 확정하고 WAV 순서는 그 뒤에 정합니다.

In [ ]:
ENV_COLUMNS = ['hive temp', 'hive humidity', 'hive pressure',
               'weather temp', 'weather pressure']


SEQUENCE_LENGTH = 6


MAX_ENV_GAP_HOURS = 2.0


SPLITS = ('train', 'val', 'test')


def fingerprint(value):
    """Stable cache/provenance key, including missing numbers and timestamps."""
    def convert(item):
        if isinstance(item, dict):
            return {str(k): convert(v) for k, v in item.items()}
        if isinstance(item, (list, tuple, np.ndarray)):
            return [convert(v) for v in item]
        if isinstance(item, (pd.Timestamp, Path)):
            return str(item)
        if isinstance(item, np.generic):
            return convert(item.item())
        if item is None or pd.isna(item):
            return None
        return item
    payload = json.dumps(convert(value), sort_keys=True, ensure_ascii=False,
                         allow_nan=False, separators=(',', ':'))
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def hive_key(value):
    if value is None or pd.isna(value) or str(value).strip().lower() in ('', 'nan', 'none'):
        raise ValueError('Missing hive identifier')
    text = str(value).strip()
    try:
        number = float(text)
    except ValueError:
        return text
    if not np.isfinite(number) or number <= 0 or not number.is_integer():
        raise ValueError('Invalid hive identifier')
    return str(int(number))


def timestamp(value):
    value = pd.Timestamp(value)
    if pd.isna(value) or value.tzinfo is not None:
        raise ValueError('A valid local timestamp without timezone is required')
    return value


def prepare_metadata(raw):
    required = {'file name', 'hive number', 'date', 'queen status', *ENV_COLUMNS}
    if required - set(raw):
        raise ValueError(f'Missing CSV columns: {required - set(raw)}')
    rows, exclusions = [], []
    for index, row in raw.iterrows():
        try:
            name = row['file name']
            if pd.isna(name) or not str(name).strip():
                raise ValueError('missing source_id')
            hive = float(row['hive number'])
            if not np.isfinite(hive) or hive <= 0 or not hive.is_integer():
                raise ValueError('invalid hive identifier')
            observed = timestamp(row['date'])
            record = {'csv_row': int(index), 'source_id': Path(str(name)).stem,
                      'hive_id': hive_key(hive), 'observed_at': observed,
                      'session_day': observed.strftime('%Y-%m-%d'),
                      'label': pd.to_numeric(row['queen status'], errors='coerce')}
            record.update({column: pd.to_numeric(row[column], errors='coerce')
                           for column in ENV_COLUMNS})
            rows.append(record)
        except (ValueError, TypeError, OverflowError) as error:
            exclusions.append({'csv_row': index, 'source_id': row.get('file name'),
                               'reason': str(error)})
    result = pd.DataFrame(rows)
    if result.empty:
        raise ValueError('No valid original environment observations')
    # An ambiguous source filename cannot be linked to a WAV safely.
    for source, group in result.groupby('source_id', sort=False):
        if len(group.drop(columns='csv_row').drop_duplicates()) != 1:
            raise ValueError(f'Conflicting CSV source: {source}')
    return result.drop_duplicates('source_id').reset_index(drop=True), pd.DataFrame(
        exclusions, columns=['csv_row', 'source_id', 'reason'])


def register_observations(frame):
    """Idempotent (hive_id, source_id) registry; conflicting measurements are errors."""
    frame = frame.copy()
    required = {'hive_id', 'source_id', 'observed_at', 'split', *ENV_COLUMNS}
    if required - set(frame):
        raise ValueError(f'Missing observation columns: {required - set(frame)}')
    frame['hive_id'] = frame['hive_id'].map(hive_key)
    frame['observed_at'] = frame['observed_at'].map(timestamp)
    if frame['source_id'].isna().any() or frame['source_id'].astype(str).str.strip().eq('').any():
        raise ValueError('Missing source_id')
    frame['source_id'] = frame['source_id'].astype(str)
    if not frame['split'].isin([*SPLITS, 'upload']).all():
        raise ValueError('Invalid observation split')
    for column in ENV_COLUMNS:
        frame[column] = pd.to_numeric(frame[column], errors='coerce').replace([np.inf, -np.inf], np.nan)
    compare = ['observed_at', 'split', *ENV_COLUMNS]
    for key, group in frame.groupby(['hive_id', 'source_id'], sort=False):
        if len(group[compare].drop_duplicates()) > 1:
            raise ValueError(f'Conflicting environment observation: {key}')
    return frame.drop_duplicates(['hive_id', 'source_id']).sort_values(
        ['hive_id', 'split', 'observed_at', 'source_id']).reset_index(drop=True)

In [ ]:
def save_json(path, value):
    def convert(item):
        if isinstance(item, np.generic):
            return item.item()
        if isinstance(item, np.ndarray):
            return item.tolist()
        if isinstance(item, (Path, pd.Timestamp)):
            return str(item)
        raise TypeError(type(item).__name__)
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False,
                                   allow_nan=False, default=convert), encoding='utf-8')


def scan_project(root, output_dir):
    root, output_dir = Path(root), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    raw = pd.read_csv(root / 'dataset/all_data_updated.csv')
    metadata, invalid = prepare_metadata(raw)
    index = metadata.set_index('source_id')
    records, excluded = [], []
    wav_paths = sorted((root / 'dataset').rglob('*.wav'))
    for path in wav_paths:
        relative = path.relative_to(root).as_posix()
        matched = re.fullmatch(r'(.+)__segment(\d+)', path.stem)
        source = matched.group(1) if matched else path.stem
        segment = int(matched.group(2)) if matched else 0
        try:
            if source not in index.index:
                raise ValueError('CSV match missing or invalid observation identifier/time')
            row = index.loc[source]
            if pd.isna(row['label']) or row['label'] not in LABEL_MAP:
                raise ValueError('Invalid queen status label')
            info = sf.info(path)
            if not 0 < info.duration <= 60.0:
                raise ValueError('WAV must contain >0 and <=60 seconds')
            record = row.to_dict()
            record.update({'source_id': source, 'file_path': relative,
                           'label': int(row['label']), 'segment': segment,
                           'recorded_at': row['observed_at'] + pd.Timedelta(seconds=60 * segment),
                           'duration': info.duration, 'sample_rate': info.samplerate,
                           'channels': info.channels, 'subtype': info.subtype})
            records.append(record)
        except (OSError, RuntimeError, ValueError) as error:
            excluded.append({'file_path': relative, 'reason': str(error)})
    pd.DataFrame(excluded, columns=['file_path', 'reason']).to_csv(
        output_dir / 'matching_exclusions.csv', index=False)
    invalid.to_csv(output_dir / 'environment_exclusions.csv', index=False)
    if not records:
        raise ValueError('No usable WAVs in dataset')
    manifest = pd.DataFrame(records)
    if not manifest['file_path'].is_unique:
        raise ValueError('Duplicate WAV paths')
    return manifest, metadata

In [ ]:
def hive_split(frame, held_hive, validation_hive=None):
    result = frame.copy()
    if validation_hive is None:
        result['split'] = np.where(result['hive_id'].eq(held_hive), 'test', 'train')
    else:
        result = result[result['hive_id'].ne(held_hive)].copy()
        result['split'] = np.where(result['hive_id'].eq(validation_hive), 'val', 'train')
    return result

def split_plan(manifest):
    if set(manifest['hive_id']) != set(HIVES):
        raise ValueError(f'Unexpected local hive IDs: {sorted(manifest["hive_id"].unique())}')
    for held in HIVES:
        for validation in HIVES:
            if validation != held:
                yield f'outer_hive_{held}/inner_hive_{validation}', hive_split(manifest, held, validation)
        yield f'outer_hive_{held}/final', hive_split(manifest, held)

def split_diagnostics(frame):
    return {}

def split_counts(frame, stage):
    rows = []
    for split, part in frame.groupby('split', sort=False):
        sources = part.drop_duplicates('source_id')
        row = {'experiment': EXPERIMENT.upper(), 'architecture': 'GAP',
               'split_definition': DEFINITIONS[EXPERIMENT], 'stage': stage, 'split': split,
               'wavs': len(part), 'sources': len(sources),
               'classes': ','.join(map(str, sorted(part['label'].unique())))}
        for label in range(4):
            row[f'wavs_class_{label}'] = int(part['label'].eq(label).sum())
            row[f'sources_class_{label}'] = int(sources['label'].eq(label).sum())
        rows.append(row)
    return rows

def validate_split(frame, stage):
    if (frame['split'].isna().any() or frame.groupby('source_id')['split'].nunique().ne(1).any()
            or not frame['file_path'].is_unique):
        raise ValueError(f'{stage}: invalid source or WAV split assignment')
    expected = ({'train', 'test'} if stage.endswith('/final') else {'train', 'val'}
                if EXPERIMENT == 'a' else set(SPLITS))
    if set(frame['split']) != expected:
        raise ValueError(f'{stage}: missing splits {expected - set(frame["split"])}')
    for split in expected:
        present = set(frame.loc[frame['split'].eq(split), 'label'])
        if (split == 'train' or EXPERIMENT != 'a') and present != set(LABEL_MAP):
            raise ValueError(f'{stage}/{split}: missing classes {sorted(set(LABEL_MAP) - present)}; see {OUTPUT}')
    if EXPERIMENT == 'a' and frame.groupby('hive_id')['split'].nunique().ne(1).any():
        raise ValueError(f'{stage}: a hive spans splits')

def audit_plan(manifest, stem):
    rows = []
    for stage, frame in split_plan(manifest):
        validate_split(frame, stage)
        rows.extend(split_counts(frame, stage))
        for _, part in frame.groupby('split', sort=False):
            ordered_wavs(part)
        checks = split_diagnostics(frame)
        if checks:
            save_json(OUTPUT / f'{stem}_diagnostics.json', checks['summary'])
            checks['detail'].to_csv(OUTPUT / f'{stem}_neighbors.csv', index=False)
    table = pd.DataFrame(rows)
    table.to_csv(OUTPUT / f'{stem}_distribution.csv', index=False)
    return table

def ordered_wavs(frame):
    frame = frame.sort_values('file_path').reset_index(drop=True)
    if EXPERIMENT == 'b':
        order = np.random.default_rng(SEED).permutation(len(frame))
        shuffled = frame.iloc[order].reset_index(drop=True)
        keys = frame[['file_path', 'source_id', 'label']].to_numpy()
        np.testing.assert_array_equal(shuffled[['file_path', 'source_id', 'label']], keys[order])
        return shuffled
    return frame

## 2. 품질 검사, 로그멜 캐시와 학습용 전처리

캐시는 파일 키와 배열의 행 순서를 대조한 뒤 읽기 전용으로 사용합니다. 음성·환경 통계와 클래스 가중치는 단계별 train 자료에서 계산합니다.

In [ ]:
LABEL_MAP = {label: f"queen status {label}" for label in range(4)}


@dataclass(frozen=True)
class AudioConfig:
    sample_rate: int = 16000
    segment_seconds: float = 60.0  # WAV 하나 전체를 모델 입력 하나로 사용한다.
    n_mels: int = 64
    n_fft: int = 1024
    hop_length: int = 512
    fmin: float = 0.0
    fmax: float = 8000.0
    top_db: float = 80.0

    def __post_init__(self):
        if min(self.sample_rate, self.segment_seconds, self.n_mels,
               self.n_fft, self.hop_length) <= 0:
            raise ValueError("전처리 크기와 시간은 양수여야 합니다.")
        if not 0 <= self.fmin < self.fmax <= self.sample_rate / 2:
            raise ValueError("멜 주파수 범위를 확인하세요.")

    @property
    def samples(self):
        return round(self.sample_rate * self.segment_seconds)

    @property
    def shape(self):
        return (self.n_mels, 1 + self.samples // self.hop_length, 1)


def read_audio(source, config, offset=0.0, duration=None):
    """WAV의 지정 구간을 mono/고정 샘플링 주파수로 읽는다. 패딩은 여기서 하지 않는다."""
    if offset < 0 or (duration is not None and duration <= 0):
        raise ValueError("시작 시각과 구간 길이를 확인하세요.")
    if hasattr(source, "seek"):
        source.seek(0)
    with sf.SoundFile(source) as audio:
        original_sr = audio.samplerate
        waveform = audio.read(dtype="float32", always_2d=True)
    if not len(waveform):
        raise ValueError("읽을 수 있는 음성 구간이 없습니다.")
    if not np.isfinite(waveform).all():
        raise ValueError("음성에 NaN 또는 무한대 값이 있습니다.")
    waveform = waveform.mean(axis=1)
    if original_sr != config.sample_rate:
        waveform = librosa.resample(
            waveform, orig_sr=original_sr, target_sr=config.sample_rate,
            res_type="soxr_hq",
        )
    # 먼저 동일한 전체 녹음을 리샘플링한 뒤 자른다. 학습/앱 구간 경계도 같아진다.
    start = round(offset * config.sample_rate)
    end = None if duration is None else start + round(duration * config.sample_rate)
    waveform = waveform[start:end]
    if not len(waveform):
        raise ValueError("읽을 수 있는 음성 구간이 없습니다.")
    return np.asarray(waveform, dtype="float32")


def log_mel(waveform, config):
    """녹음 전체를 (mel, time, channel) 로그 멜 배열로 변환한다."""
    waveform = np.asarray(waveform, dtype="float32")
    if waveform.ndim != 1 or not len(waveform) or not np.isfinite(waveform).all():
        raise ValueError("비어 있지 않은 유한한 mono 음성이 필요합니다.")
    if len(waveform) > config.samples:
        raise ValueError("녹음 길이가 모델 입력 길이와 맞지 않습니다.")
    waveform = librosa.util.fix_length(waveform, size=config.samples)
    mel = librosa.feature.melspectrogram(
        y=waveform, sr=config.sample_rate, n_fft=config.n_fft,
        hop_length=config.hop_length, n_mels=config.n_mels,
        fmin=config.fmin, fmax=config.fmax, power=2.0,
        center=True, pad_mode="constant",
    )
    return librosa.power_to_db(mel, ref=1.0, top_db=config.top_db)[..., None].astype("float32")


def audio_quality(waveform):
    """RMS·최대 진폭·±1 이상 샘플 비율을 계산한다."""
    return {
        "rms": float(np.sqrt(np.mean(np.square(waveform, dtype="float64")))),
        "peak": float(np.max(np.abs(waveform))),
        "over_full_scale_ratio": float(np.mean(np.abs(waveform) >= 1.0)),
    }


def config_dict(config):
    return asdict(config)

In [ ]:
class ObservationHistory:
    def __init__(self, observations):
        self.table = register_observations(observations)
        self.groups, self.positions = {}, {}
        for key, group in self.table.groupby(['hive_id', 'split'], sort=False):
            group = group.reset_index(drop=True)
            self.groups[key] = group
            for index, source in enumerate(group['source_id']):
                self.positions[(*key, source)] = index

    def window(self, current, length=SEQUENCE_LENGTH, max_gap=MAX_ENV_GAP_HOURS):
        key = (hive_key(current['hive_id']), str(current['split']))
        position = self.positions.get((*key, str(current['source_id'])))
        if position is None:
            raise ValueError('Current source_id is absent from the environment history')
        group = self.groups[key]
        endpoint = group.iloc[position]
        observed = timestamp(current['observed_at'])
        recorded = timestamp(current.get('recorded_at', observed))
        if observed != endpoint['observed_at'] or recorded < observed:
            raise ValueError('Current observation/recording timestamp mismatch')
        for column in ENV_COLUMNS:
            supplied, stored = pd.to_numeric(current[column], errors='coerce'), endpoint[column]
            if not np.isfinite(supplied):
                supplied = np.nan
            if not ((pd.isna(supplied) and pd.isna(stored)) or supplied == stored):
                raise ValueError(f'Current environment mismatch: {column}')
        # Inspect original adjacent gaps BEFORE cutting the six-observation window.
        start = position
        while start > 0:
            gap = (group.iloc[start]['observed_at'] - group.iloc[start - 1]['observed_at']).total_seconds() / 3600
            if gap > max_gap:
                break
            start -= 1
        start = max(start, position - length + 1)
        selected = group.iloc[start:position + 1].copy()
        delta = selected['observed_at'].diff().dt.total_seconds().fillna(0).to_numpy() / 3600
        return selected, delta


def numeric_values(frame):
    values = frame[ENV_COLUMNS].apply(pd.to_numeric, errors='coerce').to_numpy(dtype='float64')
    values[~np.isfinite(values)] = np.nan
    values[values[:, 4] == 0, 4] = np.nan
    return values


def safe_std(values):
    std = np.std(values, axis=0)
    return np.where(np.isfinite(std) & (std > 1e-8), std, 1.0)


def fit_environment_preprocessing(observations):
    observations = register_observations(observations)
    train = observations[observations['split'].eq('train')]
    if train.empty:
        raise ValueError('Training environment observations are required')
    values = numeric_values(train)
    # An entirely missing training column has deterministic 0/1 fallback statistics.
    median = np.array([np.nanmedian(c) if np.isfinite(c).any() else 0.0 for c in values.T])
    filled = np.where(np.isnan(values), median, values)
    history = ObservationHistory(train)
    deltas = np.concatenate([history.window(row)[1] for _, row in train.iterrows()])
    return {
        'version': 1, 'columns': ENV_COLUMNS, 'median': median.tolist(),
        'mean': filled.mean(axis=0).tolist(), 'std': safe_std(filled).tolist(),
        'all_missing_columns': [c for c, v in zip(ENV_COLUMNS, values.T) if not np.isfinite(v).any()],
        'time_part_categories': list(range(6)), 'current_dim': 11,
        'sequence_length': 6, 'sequence_dim': 12, 'max_gap_hours': 2.0,
        'padding_value': 0.0, 'padding': 'right',
        'delta_mean': float(deltas.mean()), 'delta_std': float(safe_std(deltas)),
        'delta_statistic_policy': 'one causal window per unique training observation; first delta=0',
        'training_observations': len(train), 'delta_values': len(deltas),
        'training_observations_sha256': fingerprint(train[['hive_id', 'source_id', 'observed_at', *ENV_COLUMNS]].to_dict('records')),
    }


def validate_preprocessing(settings):
    if (settings['columns'] != ENV_COLUMNS or settings['time_part_categories'] != list(range(6))
            or settings['current_dim'] != 11 or settings['sequence_dim'] != 12
            or settings['sequence_length'] != 6 or settings['max_gap_hours'] != 2.0
            or settings['padding_value'] != 0 or settings['padding'] != 'right'):
        raise ValueError('Incompatible environment preprocessing schema')
    for field in ('median', 'mean', 'std'):
        value = np.asarray(settings[field])
        if value.shape != (5,) or not np.isfinite(value).all():
            raise ValueError(f'Invalid preprocessing statistics: {field}')
    if (np.any(np.asarray(settings['std']) <= 0) or settings['delta_std'] <= 0
            or not np.isfinite([settings['delta_mean'], settings['delta_std']]).all()):
        raise ValueError('Invalid standard deviation')


def transform_current(frame, settings, time_column='recorded_at'):
    validate_preprocessing(settings)
    values = numeric_values(frame)
    values = np.where(np.isnan(values), settings['median'], values)
    numeric = (values - settings['mean']) / settings['std']
    parts = np.array([timestamp(t).hour // 4 for t in frame[time_column]], dtype=int)
    return np.concatenate([numeric, np.eye(6)[parts]], axis=1).astype('float32')


def make_environment_inputs(recordings, observations, settings):
    """Exactly the same entry point is used by training, uploads and replay."""
    history = observations if isinstance(observations, ObservationHistory) else ObservationHistory(observations)
    current = transform_current(recordings, settings)
    sequences = np.zeros((len(recordings), 6, 12), dtype='float32')
    counts = []
    for index, (_, row) in enumerate(recordings.iterrows()):
        selected, delta = history.window(row)
        # Copy only; never mutate original observations when segment time crosses a time bin.
        selected['recorded_at'] = selected['observed_at']
        selected.loc[selected.index[-1], 'recorded_at'] = timestamp(row['recorded_at'])
        transformed = transform_current(selected, settings)
        normalized_delta = ((delta - settings['delta_mean']) / settings['delta_std']).astype('float32')
        count = len(selected)
        sequences[index, :count] = np.column_stack([transformed, normalized_delta])
        counts.append(count)
    return current, sequences, np.asarray(counts, dtype='int64')

In [ ]:
_MAPPINGS = []


_DATASETS = []


def release_feature_file(path, namespace=None, split=None):
    """Invalidate readers before same-path replacement (including dual-input generators).

    Do not access old arrays after invalidation. Notebook passes globals() to remove
    arrays, views, datasets and dictionaries retaining earlier memmaps.
    """
    target = Path(path).resolve()
    def refers(value, seen=None):
        seen = set() if seen is None else seen
        if id(value) in seen:
            return False
        seen.add(id(value))
        if isinstance(value, np.ndarray):
            while isinstance(value, np.ndarray):
                if (isinstance(value, np.memmap) and value.filename is not None
                        and Path(value.filename).resolve() == target):
                    return True
                value = value.base
        if isinstance(value, dict):
            return any(refers(v, seen) for v in value.values())
        if isinstance(value, (list, tuple)):
            return any(refers(v, seen) for v in value)
        return False
    for reference, dependencies in _DATASETS:
        state = reference()
        if state is not None and target in dependencies:
            state['active'] = False
            state['features'] = None
    if namespace is not None:
        for name, value in list(namespace.items()):
            if name.startswith('__'):
                continue
            if refers(value) or (split and (name.endswith('_ds') or name in ('datasets', 'lstm_datasets'))):
                namespace.pop(name, None)
    gc.collect()
    # Explicit weak registry plus GC finds numpy views whose base owns the mapping.
    candidates = [ref() for ref in _MAPPINGS] + gc.get_objects()
    for value in candidates:
        try:
            if isinstance(value, np.memmap) and refers(value):
                mapping = value._mmap
                if mapping is not None and not mapping.closed:
                    mapping.close()
        except ReferenceError:
            pass


def _track(array):
    _MAPPINGS.append(weakref.ref(array))
    return array


def build_features(frame, split, root, artifact_dir, config=None,
                   silence_rms=1e-5, manual_exclusions=None, namespace=None):
    """Audio-only cache identity; metadata is rebuilt in the exact surviving WAV order."""
    config = config or AudioConfig()
    root, artifact_dir = Path(root), Path(artifact_dir)
    feature_dir = artifact_dir / 'features'
    feature_dir.mkdir(parents=True, exist_ok=True)
    feature_path = feature_dir / f'{split}.npy'
    settings_path = feature_dir / f'{split}_config.json'
    quality_path = artifact_dir / f'quality_{split}.csv'
    settings = {'audio': config_dict(config), 'silence_rms': silence_rms,
                'manual_exclusions': manual_exclusions or {}}
    frame = frame.reset_index(drop=True)
    cached, quality = None, None
    try:
        cached = json.loads(settings_path.read_text(encoding='utf-8'))
        quality = pd.read_csv(quality_path)
    except (OSError, ValueError):
        pass
    reuse = (feature_path.exists() and cached == settings and quality is not None
             and {'file_path', 'reason'}.issubset(quality)
             and quality['file_path'].tolist() == frame['file_path'].tolist())
    # New caches detect content replacement; legacy caches retain the original list policy.
    signatures = [f'{(root / p).stat().st_size}:{(root / p).stat().st_mtime_ns}' for p in frame['file_path']]
    if reuse and 'wav_stamp' in quality:
        reuse = quality['wav_stamp'].tolist() == signatures
    if reuse:
        features = _track(np.load(feature_path, mmap_mode='r', allow_pickle=False))
        if features.shape != (len(frame), *config.shape) or features.dtype != np.float32:
            release_feature_file(feature_path, namespace, split)
            raise ValueError(f'{split}: corrupt/incompatible feature cache')
        valid = np.flatnonzero(quality['reason'].fillna('').eq(''))
        # Cache storage is compacted; do not use original frame indices to index it.
        recordings = frame.iloc[valid].reset_index(drop=True)
        print(f'{split}: reuse {len(recordings)} whole-WAV features', flush=True)
    else:
        release_feature_file(feature_path, namespace, split)
        settings_path.unlink(missing_ok=True)
        features = _track(np.lib.format.open_memmap(feature_path, mode='w+', dtype='float32',
                              shape=(len(frame), *config.shape)))
        valid, reports = [], []
        for index, row in frame.iterrows():
            report = {'file_path': row['file_path'], 'source_id': row['source_id'],
                      'wav_stamp': signatures[index], 'split': split, 'reason': ''}
            try:
                if Path(row['file_path']).name in settings['manual_exclusions']:
                    raise ValueError(settings['manual_exclusions'][Path(row['file_path']).name])
                waveform = read_audio(root / row['file_path'], config)
                report.update(audio_quality(waveform))
                if report['rms'] < silence_rms:
                    raise ValueError('near-silent recording')
                features[len(valid)] = log_mel(waveform, config)
                valid.append(index)
            except (OSError, RuntimeError, ValueError) as error:
                report['reason'] = str(error)
            reports.append(report)
            if (index + 1) % 200 == 0:
                print(f'{split}: {index + 1}/{len(frame)}', flush=True)
        features.flush()
        pd.DataFrame(reports).to_csv(quality_path, index=False)
        recordings = frame.iloc[valid].reset_index(drop=True)
        save_json(settings_path, settings)
        # Read-only on both paths, to avoid accidental downstream mutation.
        features._mmap.close()
        features = _track(np.load(feature_path, mmap_mode='r', allow_pickle=False))
    recordings['feature_index'] = np.arange(len(recordings))
    recordings['offset_seconds'] = 0.0
    recordings['duration_seconds'] = recordings['duration']
    return features[:len(recordings)], recordings['label'].to_numpy(dtype='int64'), recordings


def audio_statistics(features, batch_size=8):
    total = squares = 0.0
    count = 0
    for start in range(0, len(features), batch_size):
        batch = np.asarray(features[start:start + batch_size], dtype='float64')
        total += batch.sum()
        squares += np.square(batch).sum()
        count += batch.size
    if not count:
        raise ValueError('No training audio')
    mean = float(total / count)
    return {'mean': mean, 'variance': max(float(squares / count - mean ** 2), 1e-6)}


class BatchState(dict):
    pass


def make_dataset(features, labels, env_sequence=None, training=False, batch_size=8, seed=42):
    import tensorflow as tf
    labels = np.asarray(labels, dtype='int64')
    if len(features) != len(labels) or (env_sequence is not None and len(env_sequence) != len(labels)):
        raise ValueError('Audio/environment/label alignment mismatch')
    state = BatchState(active=True, features=features)
    dependencies = set()
    base = features
    while isinstance(base, np.ndarray):
        if isinstance(base, np.memmap):
            dependencies.add(Path(base.filename).resolve())
        base = base.base
    _DATASETS.append((weakref.ref(state), dependencies))
    rng = np.random.default_rng(seed)
    def batches():
        if not state['active']:
            raise RuntimeError('Feature cache changed; rebuild datasets before training')
        order = rng.permutation(len(labels)) if training else np.arange(len(labels))
        for start in range(0, len(order), batch_size):
            if not state['active']:
                raise RuntimeError('Feature cache changed; rebuild datasets')
            indices = order[start:start + batch_size]
            audio = np.asarray(state['features'][indices], dtype='float32')
            inputs = audio if env_sequence is None else {
                'audio': audio, 'env_sequence': np.asarray(env_sequence[indices], dtype='float32')}
            yield inputs, labels[indices]
    spec = tf.TensorSpec((None, *features.shape[1:]), tf.float32)
    if env_sequence is not None:
        spec = {'audio': spec, 'env_sequence': tf.TensorSpec((None, 6, 12), tf.float32)}
    dataset = tf.data.Dataset.from_generator(batches, output_signature=(spec, tf.TensorSpec((None,), tf.int64)))
    dataset = dataset.apply(tf.data.experimental.assert_cardinality(math.ceil(len(labels) / batch_size)))
    options = tf.data.Options()
    options.threading.private_threadpool_size = 2
    return dataset.with_options(options).prefetch(1)

In [ ]:
class IndexedAudio:
    """Read a batch from a stable file-key index without duplicating fold-sized arrays."""
    def __init__(self, arrays, pointers, files):
        self.arrays = arrays
        self.locations = np.asarray([pointers[file] for file in files], dtype='int64')
        self.shape = (len(files), *CONFIG.shape)

    def __len__(self):
        return self.shape[0]

    def __getitem__(self, key):
        locations = self.locations[key]
        if locations.ndim == 1:
            shard, row = locations
            return self.arrays[shard][row]
        output = np.empty((len(locations), *self.shape[1:]), dtype=np.float32)
        for shard in np.unique(locations[:, 0]):
            selected = locations[:, 0] == shard
            output[selected] = self.arrays[shard][locations[selected, 1]]
        return output

def feature_bank(manifest):
    legacy = ROOT / 'artifacts/recording_60s'
    settings = {'audio': config_dict(CONFIG), 'silence_rms': SILENCE_RMS,
                'manual_exclusions': MANUAL_EXCLUSIONS}
    stamps = {}
    for file in manifest['file_path']:
        info = (ROOT / file).stat()
        stamps[file] = f'{info.st_size}:{info.st_mtime_ns}'
    arrays, pointers, quality_rows = [], {}, {}
    cache_notes = {}
    if (legacy / 'manifest.csv').is_file():
        old_manifest = pd.read_csv(legacy / 'manifest.csv', usecols=['file_path', 'split'])
        if not old_manifest['file_path'].is_unique:
            raise ValueError('Legacy feature manifest has duplicate WAV keys')
        for split in SPLITS:
            features_dir = legacy / 'features'
            try:
                old_settings = json.loads((features_dir / f'{split}_config.json').read_text(encoding='utf-8'))
                quality = pd.read_csv(legacy / f'quality_{split}.csv').fillna({'reason': ''})
                original_order = old_manifest.loc[old_manifest['split'].eq(split), 'file_path'].tolist()
                if (old_settings != settings or not {'file_path', 'reason', 'wav_stamp'}.issubset(quality)
                        or quality['file_path'].tolist() != original_order):
                    raise ValueError('incompatible settings, WAV order, or file stamps')
                array = np.load(features_dir / f'{split}.npy', mmap_mode='r', allow_pickle=False)
                if array.shape != (len(quality), *CONFIG.shape) or array.dtype != np.float32:
                    raise ValueError(f'incompatible array: {array.shape}, {array.dtype}')
            except (OSError, ValueError, KeyError) as error:
                cache_notes[split] = str(error)
                continue
            shard = len(arrays)
            arrays.append(array)
            valid_index = 0  # The original writer compacts successful rows.
            for row in quality.to_dict('records'):
                file = row['file_path']
                if stamps.get(file) == row['wav_stamp']:
                    row['feature_origin'] = f'legacy/{split}'
                    quality_rows[file] = row
                    if not row['reason']:
                        pointers[file] = (shard, valid_index)
                valid_index += not bool(row['reason'])
            cache_notes[split] = 'compatible'
    missing = manifest[~manifest['file_path'].isin(quality_rows)].sort_values('file_path')
    if not missing.empty:
        print(f'Extracting {len(missing)} WAVs with no compatible cached row', flush=True)
        array, _, valid = build_features(missing, 'uncached', ROOT, OUTPUT, CONFIG,
                                         SILENCE_RMS, MANUAL_EXCLUSIONS)
        shard = len(arrays)
        arrays.append(array)
        pointers.update({file: (shard, index) for index, file in enumerate(valid['file_path'])})
        quality = pd.read_csv(OUTPUT / 'quality_uncached.csv').fillna({'reason': ''})
        quality['feature_origin'] = 'extracted'
        quality_rows.update(quality.set_index('file_path', drop=False).to_dict('index'))
    quality = pd.DataFrame([quality_rows[file] for file in manifest['file_path']])
    quality = quality.drop(columns='split')
    quality.to_csv(OUTPUT / 'audio_quality.csv', index=False)
    valid = manifest[manifest['file_path'].isin(pointers)].copy()
    save_json(OUTPUT / 'audio_cache_audit.json', {
        'valid_wavs': len(valid), 'excluded_wavs': len(manifest) - len(valid),
        'legacy_shards': cache_notes, 'uncached_wavs': len(missing),
        'wav_origin_counts': quality['feature_origin'].value_counts().to_dict(),
    })
    return arrays, pointers, valid

In [ ]:
def select_smoke(frame):
    selected = (frame.sort_values(['observed_at', 'file_path']).drop_duplicates('source_id')
                .groupby(['split', 'label'], sort=False, group_keys=False).head(SMOKE_PER_CLASS))
    if EXPERIMENT == 'a':
        train_hives = selected.loc[selected['split'].eq('train'), 'hive_id']
        missing_hives = frame[frame['split'].eq('train') & ~frame['hive_id'].isin(train_hives)]
        selected = pd.concat([selected, missing_hives.sort_values('file_path').drop_duplicates('hive_id')])
    if EXPERIMENT == 'c':
        train = frame[frame['split'].eq('train')].drop_duplicates('source_id')
        hours = train['observed_at'].dt.floor('h')
        partners = []
        for row in selected[~selected['split'].eq('train')].itertuples():
            partner = train[hours.eq(row.observed_at.floor('h')) & train['hive_id'].ne(row.hive_id)]
            partners.append(partner.iloc[[0]])
        selected = pd.concat([selected, *partners]).drop_duplicates('file_path')
    # Include two WAVs per selected source when available to exercise probability averaging.
    sources = set(selected['source_id'])
    return (frame[frame['source_id'].isin(sources)].sort_values('file_path')
            .groupby('source_id', sort=False, group_keys=False).head(2))

def prepare_stage(frame, observations, directory, use_sequence):
    settings = fit_environment_preprocessing(observations)
    train_obs = observations[observations['split'].eq('train')]
    history = ObservationHistory(observations) if use_sequence else None
    prepared = {}
    for split, part in frame.groupby('split', sort=False):
        recordings = ordered_wavs(part)
        files = recordings['file_path'].tolist()
        recordings['evaluation_order'] = np.arange(len(recordings))
        audio = IndexedAudio(AUDIO_ARRAYS, AUDIO_POINTERS, files)
        labels = recordings['label'].to_numpy(dtype='int64')
        if use_sequence:
            current, sequence, counts = make_environment_inputs(recordings, history, settings)
            mask = np.arange(SEQUENCE_LENGTH)[None, :] < counts[:, None]
            np.testing.assert_array_equal(np.any(sequence != 0, axis=2), mask)
            recordings['environment_history_length'] = counts
        else:
            current = transform_current(recordings, settings)
            sequence = None
        prepared[split] = {'audio': audio, 'labels': labels, 'current': current,
                           'sequence': sequence, 'recordings': recordings}
    aligned = pd.concat([part['recordings'] for part in prepared.values()], ignore_index=True)
    aligned.to_csv(directory / 'manifest.csv', index=False)
    observations.to_csv(directory / 'environment_observations.csv', index=False)
    if use_sequence:
        source_history = aligned.drop_duplicates('source_id')
        counts = pd.concat([
            aligned.groupby(['split', 'environment_history_length']).size().rename('wavs'),
            source_history.groupby(['split', 'environment_history_length']).size().rename('sources'),
        ], axis=1)
        counts.to_csv(directory / 'environment_history_lengths.csv')
    train = prepared['train']
    normalization = audio_statistics(train['audio'], BATCH_SIZE)
    weights = compute_class_weight('balanced', classes=np.arange(4), y=train['labels'])
    class_weights = dict(enumerate(map(float, weights)))
    fit_wavs = train['recordings'][['file_path', 'source_id', 'label']]
    fit_wavs.to_csv(directory / 'fit_audio_wavs.csv', index=False)
    train_obs.to_csv(directory / 'fit_environment_observations.csv', index=False)
    preprocessing = {
        'schema_version': 2, 'architecture': 'GAP', 'audio': config_dict(CONFIG),
        'input_unit': 'whole_wav', 'quality': {'silence_rms': SILENCE_RMS,
                                              'manual_exclusions': MANUAL_EXCLUSIONS},
        'normalization': normalization, 'environment': settings, 'label_map': LABEL_MAP,
        'class_weights': class_weights, 'seed': SEED, 'batch_size': BATCH_SIZE,
        'training_wavs': len(fit_wavs), 'training_wavs_sha256': fingerprint(fit_wavs.to_dict('records')),
    }
    save_json(directory / 'preprocessing.json', preprocessing)
    return prepared, preprocessing

In [ ]:
CONFIG = AudioConfig()
local_manifest, METADATA = scan_project(ROOT, OUTPUT)
if EXPERIMENT in ('b', 'c'):
    FIXED = source_assignments(local_manifest)
    FIXED.to_csv(OUTPUT / 'source_assignments.csv')
assigned_distribution = audit_plan(local_manifest, 'assigned')
AUDIO_ARRAYS, AUDIO_POINTERS, USABLE_WAVS = feature_bank(local_manifest)
USABLE_WAVS.to_csv(OUTPUT / 'usable_wavs.csv', index=False)
if EXPERIMENT == 'c':
    lost = FIXED[FIXED['split'].isin(['val', 'test']) & ~FIXED.index.isin(USABLE_WAVS['source_id'])]
    if not lost.empty:
        lost.to_csv(OUTPUT / 'excluded_held_sources.csv')
        raise ValueError(f'C: {len(lost)} held sources have no usable WAVs; see {OUTPUT}')
quality_distribution = audit_plan(USABLE_WAVS, 'quality_checked')
display(quality_distribution)

## 3. 공통 CNN과 결합 모델

GAP CNN의 64차원 음향 특징을 사용합니다. 결합 모델을 학습하는 동안 음향 인코더를 고정합니다.

In [ ]:
def make_cnn(config, normalization):
    cnn = models.Sequential(name='audio_cnn')
    cnn.add(layers.Input(shape=config.shape, name='audio_input'))
    cnn.add(layers.Normalization(axis=None, **normalization, name='audio_normalization'))
    for filters in (16, 32, 64):
        cnn.add(layers.Conv2D(filters, (3, 3), padding='same', activation='relu', name=f'conv_{filters}'))
        cnn.add(layers.MaxPooling2D((2, 2), name=f'pool_{filters}'))
    cnn.add(layers.GlobalAveragePooling2D(name='audio_global_average'))
    cnn.add(layers.Dense(64, activation='relu', name='audio_features'))
    cnn.add(layers.Dropout(0.3, name='audio_dropout'))
    cnn.add(layers.Dense(4, activation='softmax', name='audio_classification'))
    cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return cnn

In [ ]:
def make_mlp():
    mlp = models.Sequential([
        layers.Input(shape=(75,)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dense(4, activation='softmax'),
    ], name='cnn_mlp_classifier')
    mlp.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return mlp

In [ ]:
def make_lstm(config, lstm_encoder):
    audio_input = layers.Input(shape=config.shape, name='audio')
    env_input = layers.Input(shape=(6, 12), name='env_sequence')
    lstm_encoder.trainable = False
    audio_features = lstm_encoder(audio_input, training=False)
    env = layers.Masking(mask_value=0.0)(env_input)
    env = layers.LSTM(64, return_sequences=False, stateful=False)(env)
    env_features = layers.Dense(64, activation='relu')(env)
    x = layers.Concatenate()([audio_features, env_features])
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, activation='relu')(x)
    outputs = layers.Dense(4, activation='softmax')(x)
    lstm = models.Model({'audio': audio_input, 'env_sequence': env_input}, outputs,
                        name='cnn_lstm_classifier')
    lstm.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return lstm

## 4. 학습, 평가 및 저장

WAV별 확률과 원본별 평균 확률을 각각 평가합니다. 모델 간 비교는 같은 평가 WAV와 정답을 파일 키로 확인한 뒤 수행합니다.

학습된 모델로 해당 단계의 train·val·test를 각각 추론합니다. 단계마다 정확도, macro F1, 클래스별 재현율과 WAV·원본 수를 화면과 `evaluation_summary.csv`에 표시합니다. 예측과 상세 지표는 단계 아래 `<모델>/<split>/`에 저장합니다. 마지막 표와 `test_comparison.csv`는 결합 모델의 테스트 결과입니다.

내부 fold에는 train·val, 최종 재학습에는 train·test가 있습니다. 마지막 테스트 표는 외부 벌통별 결과와 외부 예측을 모은 결과를 구분합니다.

In [ ]:
def copy_audio_encoder(cnn, name='audio_encoder'):
    """Clone layers AND weights: never share layer instances between final models."""
    from keras import layers, models
    encoder = models.Sequential([layers.Input(shape=cnn.input_shape[1:])], name=name)
    for layer in cnn.layers:
        clone = layer.__class__.from_config(layer.get_config())
        encoder.add(clone)
        clone.set_weights(layer.get_weights())
        if layer.name == 'audio_features':
            break
    if encoder.output_shape[-1] != 64:
        raise ValueError('Missing 64-dimensional audio_features layer')
    encoder.trainable = False
    return encoder


def extract_audio_features(encoder, features, batch_size=8):
    return np.concatenate([np.asarray(encoder(np.asarray(features[i:i + batch_size]), training=False))
                           for i in range(0, len(features), batch_size)], axis=0)


def fit_classifier(model, train, val, class_weights, epochs=30, batch_size=8):
    from keras.callbacks import EarlyStopping
    callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    kwargs = {'batch_size': batch_size} if isinstance(train, tuple) else {}
    start = perf_counter()
    if isinstance(train, tuple):
        history = model.fit(*train, validation_data=val, epochs=epochs,
                            class_weight=class_weights, callbacks=[callback] if val is not None else [], verbose=2, **kwargs)
    else:
        history = model.fit(train, validation_data=val, epochs=epochs, shuffle=False,
                            class_weight=class_weights, callbacks=[callback] if val is not None else [], verbose=2)
    history.params['fit_seconds'] = perf_counter() - start
    history.params['epoch_seconds'] = history.params['fit_seconds'] / len(history.epoch)
    return history




def plot_learning_curves(history_table, title):
    """Plot train and validation metrics over the epochs that actually ran."""
    from matplotlib.ticker import MaxNLocator, PercentFormatter

    if 'epoch' in history_table:
        epochs = history_table['epoch'].to_numpy()
    else:
        epochs = np.arange(1, len(history_table) + 1)
    fig, (loss_ax, accuracy_ax) = plt.subplots(
        1, 2, figsize=(10.5, 3.7), sharex=True, layout='constrained')
    fig.suptitle(title, fontweight='bold')
    for ax, metric, label in [(loss_ax, 'loss', 'Loss'), (accuracy_ax, 'accuracy', 'Accuracy')]:
        ax.plot(epochs, history_table[metric], 'o-', color='#22639D', label='Train', markersize=4)
        if f'val_{metric}' in history_table:
            ax.plot(epochs, history_table[f'val_{metric}'], 's--', color='#A34D21',
                    label='Validation', markersize=4)
        ax.set(xlabel='Epoch', title=label)
        ax.grid(axis='y', alpha=0.25)
        ax.set_axisbelow(True)
    loss_ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    if len(epochs) == 1:
        loss_ax.set_xticks(epochs)
    if 'val_loss' in history_table:
        best_epoch = epochs[history_table['val_loss'].to_numpy().argmin()]
        loss_ax.axvline(best_epoch, color='#69727D', linestyle=':',
                        label=f'Lowest val loss (epoch {best_epoch})')
    accuracy_ax.set_ylim(0, 1)
    accuracy_ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    loss_ax.legend(loc='best')
    accuracy_ax.legend(loc='best')
    return fig




@dataclass
class Bundle:
    kind: str
    model: object
    encoder: object
    config: AudioConfig
    settings: dict


def load_bundle(directory):
    from keras.models import load_model
    directory = Path(directory)
    settings = json.loads((directory / 'preprocessing.json').read_text(encoding='utf-8'))
    if settings.get('schema_version') != 2:
        raise ValueError('Run the updated notebook to export the two-model schema')
    config = AudioConfig(**settings['audio'])
    if config.shape != (64, 1876, 1):
        raise ValueError('Expected a whole 60-second audio input')
    validate_preprocessing(settings['environment'])
    if {int(k): v for k, v in settings['label_map'].items()} != LABEL_MAP:
        raise ValueError('Expected numeric queen status labels 0..3')
    kind = settings['model_kind']
    encoder = None
    if kind == 'cnn_mlp':
        encoder = load_model(directory / 'audio_encoder.keras', compile=False)
        model = load_model(directory / 'mlp.keras', compile=False)
        if encoder.input_shape != (None, *config.shape) or encoder.output_shape != (None, 64):
            raise ValueError('Incompatible audio encoder')
        if model.input_shape != (None, 75):
            raise ValueError('MLP requires 75 input features')
        encoder.trainable = False
    elif kind == 'cnn_lstm':
        model = load_model(directory / 'model.keras', compile=False)
        shapes = {tensor.name.split(':')[0]: tuple(tensor.shape[1:]) for tensor in model.inputs}
        if shapes != {'audio': config.shape, 'env_sequence': (6, 12)}:
            raise ValueError('Incompatible CNN+LSTM input shapes')
    else:
        raise ValueError(f'Unknown model kind: {kind}')
    if model.output_shape != (None, 4):
        raise ValueError('Expected four class scores')
    return Bundle(kind, model, encoder, config, settings)


def input_context_key(current, observations):
    columns = ['hive_id', 'source_id', 'observed_at', 'recorded_at', 'split', *ENV_COLUMNS]
    current = {key: current[key] for key in columns}
    observations = register_observations(observations)
    return fingerprint({'current': current, 'history': observations[
        ['hive_id', 'source_id', 'observed_at', 'split', *ENV_COLUMNS]].to_dict('records')})


def prepare_inputs(current, observations, settings):
    return make_environment_inputs(pd.DataFrame([dict(current)]), observations, settings['environment'])


def predict_waveform(bundle, waveform, current, observations):
    quality = audio_quality(waveform)
    if quality['rms'] < bundle.settings['quality']['silence_rms']:
        raise ValueError('거의 무음인 녹음입니다.')
    feature = log_mel(waveform, bundle.config)
    env, sequence, counts = prepare_inputs(current, observations, bundle.settings)
    if bundle.kind == 'cnn_mlp':
        audio = np.asarray(bundle.encoder(feature[None], training=False))
        inputs = np.concatenate([audio, env], axis=1)
    else:
        inputs = {'audio': feature[None], 'env_sequence': sequence}
    scores = np.asarray(bundle.model(inputs, training=False))
    if scores.shape != (1, 4) or not np.isfinite(scores).all():
        raise ValueError('Invalid model scores')
    return scores[0], feature, int(counts[0])

In [ ]:
def predict_wavs(model, inputs, part):
    recordings = part['recordings'].copy()
    np.testing.assert_array_equal(recordings['label'], part['labels'])
    scores = np.asarray(model.predict(inputs, verbose=0))
    if scores.shape != (len(recordings), 4) or not np.isfinite(scores).all():
        raise ValueError('Invalid predictions or prediction count')
    np.testing.assert_allclose(scores.sum(axis=1), 1.0, atol=1e-6)
    recordings[SCORE_COLUMNS] = scores
    recordings['predicted_label'] = scores.argmax(axis=1)
    return recordings

def evaluate_predictions(wavs, directory, require_all=False):
    from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix
    from sklearn.metrics import precision_recall_fscore_support

    directory.mkdir(parents=True, exist_ok=False)
    if not wavs['file_path'].is_unique:
        raise ValueError('Duplicate evaluation WAV key')
    group = wavs.groupby('source_id', sort=True)
    identity = ['hive_id', 'observed_at', 'session_day', 'label', 'split']
    if group[identity].nunique().ne(1).any().any():
        raise ValueError('Evaluation source identity mismatch')
    sources = group[identity].first().join(group[SCORE_COLUMNS].mean())
    sources['wav_count'] = group.size()
    sources['predicted_label'] = sources[SCORE_COLUMNS].to_numpy().argmax(axis=1)
    sources = sources.reset_index()
    wavs.sort_values(['recorded_at', 'file_path']).to_csv(directory / 'wav_predictions.csv', index=False)
    sources.to_csv(directory / 'source_predictions.csv', index=False)
    metrics = {}
    for unit, frame in [('wav', wavs), ('source', sources)]:
        truth, prediction = frame['label'], frame['predicted_label']
        labels = np.arange(4)
        precision, recall, f1, support = precision_recall_fscore_support(
            truth, prediction, labels=labels, zero_division=0)
        present = support > 0
        if require_all and not present.all():
            raise ValueError(f'{unit}: expected all four classes in pooled outer evaluation')
        table = pd.DataFrame({'label': labels, 'precision': precision, 'recall': recall,
                              'f1': f1, 'support': support})
        table.loc[~present, ['precision', 'recall', 'f1']] = np.nan
        table.to_csv(directory / f'{unit}_class_report.csv', index=False, na_rep='N/A')
        matrix = confusion_matrix(truth, prediction, labels=labels)
        pd.DataFrame(matrix, index=labels, columns=labels).to_csv(
            directory / f'{unit}_confusion_matrix.csv', index_label='true_label')
        plot = ConfusionMatrixDisplay(matrix, display_labels=labels).plot(cmap='Blues', colorbar=False)
        plot.ax_.set_title(f'{unit.capitalize()} confusion matrix')
        plot.figure_.savefig(directory / f'{unit}_confusion_matrix.png', dpi=140, bbox_inches='tight')
        plt.close(plot.figure_)
        metrics[unit] = {'accuracy': float(accuracy_score(truth, prediction)),
                         'macro_f1': float(f1[present].mean()), 'support': len(frame),
                         'classes': labels[present].tolist(), 'class_support': support.tolist(),
                         'class_recall': {str(i): float(recall[i]) if present[i] else 'N/A' for i in labels}}
    save_json(directory / 'metrics.json', metrics)
    return metrics

### 모델 체크포인트 저장

각 모델 학습이 끝나면 Keras 모델과 전처리 설정을 해당 단계 폴더에 저장합니다. 학습 이력과 예측 파일도 단계가 끝날 때마다 기록합니다.

In [ ]:
def save_stage_model(model, name, directory, preprocessing):
    save_json(directory / 'preprocessing.json', {**preprocessing, 'model_kind': name})
    model.save(directory / ('mlp.keras' if name == 'cnn_mlp' else 'model.keras'))

In [ ]:
def train_and_save(model, name, train, val, preprocessing, epochs, directory):
    directory.mkdir()
    history = fit_classifier(model, train, val, preprocessing['class_weights'], epochs, BATCH_SIZE)
    table = pd.DataFrame(history.history)
    table.insert(0, 'epoch', np.asarray(history.epoch) + 1)
    if not np.isfinite(table.to_numpy()).all():
        raise ValueError(f'{name}: non-finite training metrics')
    table.to_csv(directory / 'training_history.csv', index=False)
    best = int(table['val_loss'].to_numpy().argmin() + 1) if val is not None else None
    if val is None and len(table) != epochs:
        raise ValueError(f'{name}: expected exactly {epochs} fixed epochs')
    stats = {'epochs_requested': epochs, 'epochs_completed': len(table), 'best_val_loss_epoch': best,
             'fit_seconds': history.params['fit_seconds'], 'epoch_seconds': history.params['epoch_seconds'],
             'early_stopping': {'monitor': 'val_loss', 'patience': 5, 'restore_best_weights': True}
             if val is not None else None}
    save_json(directory / 'training_metrics.json', stats)
    plot = plot_learning_curves(table, name)
    plot.savefig(directory / 'learning_curves.png', dpi=140, bbox_inches='tight')
    plt.close(plot)
    save_stage_model(model, name, directory, preprocessing)
    return best

def check_reload(name, directory, split, part, observations, scores):
    first = part['recordings'].iloc[0]
    if name == 'cnn':
        reloaded = models.load_model(directory / 'model.keras', compile=False)
        predicted = np.asarray(reloaded(part['audio'][:1], training=False))[0]
    else:
        bundle = load_bundle(directory)
        if MODE == 'smoke':
            waveform = read_audio(ROOT / first['file_path'], CONFIG)
            predicted, feature, count = predict_waveform(bundle, waveform, first, observations)
            np.testing.assert_allclose(feature, part['audio'][0], rtol=1e-5, atol=1e-4)
            if name == 'cnn_lstm':
                assert count == first['environment_history_length']
        elif name == 'cnn_mlp':
            audio_vector = np.asarray(bundle.encoder(part['audio'][:1], training=False))
            combined = np.concatenate([audio_vector, part['current'][:1]], axis=1)
            predicted = np.asarray(bundle.model(combined, training=False))[0]
        else:
            sample = {'audio': part['audio'][:1], 'env_sequence': part['sequence'][:1]}
            predicted = np.asarray(bundle.model(sample, training=False))[0]
    difference = np.max(np.abs(predicted - scores))
    np.testing.assert_allclose(predicted, scores, rtol=1e-5, atol=2e-6)
    save_json(directory / split / 'reload_check.json', {'file_path': first['file_path'],
              'max_score_difference': difference, 'raw_wav_replay': MODE == 'smoke' and name != 'cnn'})

def fit_stage(stage, frame, observation_candidates, budgets, use_sequence):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    directory = OUTPUT / stage
    directory.mkdir(parents=True, exist_ok=False)
    if MODE == 'smoke':
        frame = select_smoke(frame)
    validate_split(frame, stage)
    checks = split_diagnostics(frame)
    if checks:
        save_json(directory / 'split_diagnostics.json', checks['summary'])
        checks['detail'].to_csv(directory / 'split_neighbors.csv', index=False)
    pd.DataFrame(split_counts(frame, stage)).to_csv(directory / 'split_distribution.csv', index=False)
    audio_sources = set(frame['source_id'])
    observations = observation_candidates.copy()
    observations['has_audio'] = observations['source_id'].isin(audio_sources)
    include = observations['has_audio']
    if EXPERIMENT == 'a':
        include = include | ~observations['source_id'].isin(LOCAL_AUDIO_SOURCES)
    observations = register_observations(observations[include])
    prepared, preprocessing = prepare_stage(frame, observations, directory, use_sequence)
    datasets = {
        split: make_dataset(part['audio'], part['labels'], training=(split == 'train'),
                            batch_size=BATCH_SIZE, seed=SEED)
        for split, part in prepared.items()
    }
    cnn = make_cnn(CONFIG, preprocessing['normalization'])
    best = {'cnn': train_and_save(cnn, 'cnn', datasets['train'], datasets.get('val'),
                                  preprocessing, budgets['cnn'], directory / 'cnn')}
    encoder = copy_audio_encoder(cnn)
    encoder_key = fingerprint(encoder.get_weights())
    preprocessing = {**preprocessing, 'shared_audio_encoder_sha256': encoder_key}
    mlp_inputs = {
        split: np.concatenate([extract_audio_features(encoder, part['audio'], BATCH_SIZE),
                               part['current']], axis=1)
        for split, part in prepared.items()
    }
    assert all(values.shape == (len(prepared[split]['labels']), 75) for split, values in mlp_inputs.items())
    mlp = make_mlp()
    val = (mlp_inputs['val'], prepared['val']['labels']) if 'val' in prepared else None
    best['cnn_mlp'] = train_and_save(mlp, 'cnn_mlp', (mlp_inputs['train'], prepared['train']['labels']),
                                     val, preprocessing, budgets['cnn_mlp'], directory / 'cnn_mlp')
    encoder.save(directory / 'cnn_mlp/audio_encoder.keras')
    assert not encoder.trainable and fingerprint(encoder.get_weights()) == encoder_key
    pipelines = {'cnn': (cnn, datasets), 'cnn_mlp': (mlp, mlp_inputs)}
    if use_sequence:
        lstm_encoder = copy_audio_encoder(cnn, name='lstm_audio_encoder')
        assert lstm_encoder is not encoder and not lstm_encoder.trainable
        assert fingerprint(lstm_encoder.get_weights()) == encoder_key
        lstm = make_lstm(CONFIG, lstm_encoder)
        lstm_datasets = {
            split: make_dataset(part['audio'], part['labels'], part['sequence'],
                                training=(split == 'train'), batch_size=BATCH_SIZE, seed=SEED)
            for split, part in prepared.items()
        }
        best['cnn_lstm'] = train_and_save(lstm, 'cnn_lstm', lstm_datasets['train'], lstm_datasets.get('val'),
                                          preprocessing, budgets['cnn_lstm'], directory / 'cnn_lstm')
        assert fingerprint(lstm_encoder.get_weights()) == encoder_key
        pipelines['cnn_lstm'] = (lstm, lstm_datasets)
    result = {}
    for split in SPLITS:
        if split not in prepared:
            continue
        results, summaries = {}, {}
        part = prepared[split]
        for name, (model, inputs) in pipelines.items():
            evaluation_input = inputs[split]
            if split == 'train' and name != 'cnn_mlp':
                # fit() shuffles WAVs each epoch. Inference follows the saved manifest order.
                sequence = part['sequence'] if name == 'cnn_lstm' else None
                evaluation_input = make_dataset(part['audio'], part['labels'], sequence,
                                                batch_size=BATCH_SIZE, seed=SEED)
            wavs = predict_wavs(model, evaluation_input, part)
            model_dir = directory / name
            summaries[name] = evaluate_predictions(wavs, model_dir / split)
            if split != 'train':
                check_reload(name, model_dir, split, part, observations,
                             wavs.iloc[0][SCORE_COLUMNS].to_numpy(dtype='float32'))
            results[name] = wavs
        pair_predictions(results, directory / f'{split}_paired_predictions.csv')
        result[split] = (results, summaries)
    summary = pd.DataFrame([
        row for split, (_, metrics) in result.items()
        for row in comparison_rows(metrics, stage, split, include_cnn=True)
    ])
    summary_path = directory / 'evaluation_summary.csv'
    summary.to_csv(summary_path, index=False, na_rep='N/A')
    print(f'{stage} | train / val / test evaluation | {summary_path}')
    display(summary[['split', 'model', 'unit', 'accuracy', 'macro_f1', 'wavs', 'sources',
                     'classes', *[f'recall_{i}' for i in range(4)]]])
    return best, result

In [ ]:
def pair_predictions(results, path):
    if 'cnn_lstm' not in results:
        return
    mlp = results['cnn_mlp'].set_index('file_path').sort_index()
    lstm = results['cnn_lstm'].set_index('file_path').sort_index()
    identity = ['source_id', 'label', 'hive_id', 'recorded_at']
    pd.testing.assert_frame_equal(mlp[identity], lstm[identity])
    predictions = ['predicted_label', *SCORE_COLUMNS]
    paired = mlp[identity].join(mlp[predictions].add_prefix('mlp_')).join(
        lstm[predictions].add_prefix('lstm_'))
    paired.to_csv(path)

def comparison_rows(metrics, scope, split, hive=None, *, include_cnn=False):
    rows = []
    for name, levels in metrics.items():
        if name == 'cnn' and not include_cnn:
            continue
        for unit, scores in levels.items():
            row = {'experiment': EXPERIMENT.upper(), 'architecture': 'GAP', 'mode': MODE,
                   'split_definition': DEFINITIONS[EXPERIMENT], 'scope': scope, 'split': split,
                   'hive_id': hive, 'model': name, 'unit': unit,
                   'wavs': levels['wav']['support'], 'sources': levels['source']['support'],
                   'accuracy': scores['accuracy'], 'macro_f1': scores['macro_f1'],
                   'classes': ','.join(map(str, scores['classes']))}
            row.update({f'wavs_class_{i}': levels['wav']['class_support'][i] for i in range(4)})
            row.update({f'sources_class_{i}': levels['source']['class_support'][i] for i in range(4)})
            row.update({f'recall_{i}': scores['class_recall'][str(i)] for i in range(4)})
            rows.append(row)
    return rows

def write_comparison(rows):
    current = pd.DataFrame(rows)
    current.to_csv(OUTPUT / 'comparison.csv', index=False, na_rep='N/A')
    test = current[current['split'].eq('test')]
    test_path = OUTPUT / 'test_comparison.csv'
    test.to_csv(test_path, index=False, na_rep='N/A')
    columns = ['experiment', 'scope', 'model', 'unit', 'accuracy', 'macro_f1',
               'wavs', 'sources', 'classes', *[f'recall_{i}' for i in range(4)]]
    if EXPERIMENT == 'a':
        columns.insert(2, 'hive_id')
    print(f'Final test comparison | {test_path}')
    display(test[columns])
    if MODE == 'full':
        tables, distributions = [], []
        for letter in 'abc':
            if letter == EXPERIMENT:
                run = OUTPUT
            else:
                root = EXPERIMENTS_DIR / f'ex_{letter}'
                complete = sorted(p.parent for p in root.glob('*/completed.json'))
                if not complete:
                    continue
                run = complete[-1]
            table = pd.read_csv(run / 'comparison.csv', keep_default_na=False)
            tables.append(table[table['split'].eq('test')].assign(result_run=run.name))
            distribution = pd.read_csv(run / 'quality_checked_distribution.csv')
            distributions.append(distribution.assign(result_run=run.name))
        combined = pd.concat(tables, ignore_index=True)
        combined_path = OUTPUT / 'experiment_comparison.csv'
        combined.to_csv(combined_path, index=False)
        pd.concat(distributions, ignore_index=True).to_csv(OUTPUT / 'experiment_split_distribution.csv', index=False)
        if len(tables) > 1:
            print(f'Test results by experiment and split definition | {combined_path}')
            display(combined[['experiment', 'split_definition', 'scope', 'hive_id',
                              'model', 'unit', 'accuracy', 'macro_f1', 'wavs', 'sources', 'classes']])

In [ ]:
def run_experiment():
    evaluation_wavs = {'cnn_mlp': [], 'cnn_lstm': []}
    hive_rows = []
    held_hives = ('3',) if MODE == 'smoke' else HIVES
    stages = {'cnn': EPOCHS, 'cnn_mlp': EPOCHS, 'cnn_lstm': EPOCHS}
    for held in held_hives:
        internal = []
        outer = OUTPUT / f'outer_hive_{held}'
        for validation in HIVES:
            if validation == held:
                continue
            best, _ = fit_stage(f'outer_hive_{held}/inner_hive_{validation}',
                                hive_split(USABLE_WAVS, held, validation),
                                hive_split(METADATA, held, validation), stages, use_sequence=True)
            internal.append({'validation_hive': validation, **best})
        choices = pd.DataFrame(internal)
        choices.to_csv(outer / 'inner_best_epochs.csv', index=False)
        budgets = {name: int(choices[name].median()) for name in stages}
        save_json(outer / 'final_epoch_budgets.json', budgets)
        _, result = fit_stage(f'outer_hive_{held}/final', hive_split(USABLE_WAVS, held),
                              hive_split(METADATA, held), budgets, use_sequence=True)
        predictions, metrics = result['test']
        hive_rows.extend(comparison_rows(metrics, 'outer_hive', 'test', held))
        for name in evaluation_wavs:
            evaluation_wavs[name].append(predictions[name].assign(outer_test_hive=held))
    hive_metrics = pd.DataFrame(hive_rows)
    hive_metrics.to_csv(OUTPUT / 'hive_metrics.csv', index=False, na_rep='N/A')
    hive_metrics.groupby(['model', 'unit'], as_index=False).agg(
        equally_weighted_hive_accuracy=('accuracy', 'mean'), hives=('hive_id', 'nunique')
    ).to_csv(OUTPUT / 'equal_hive_accuracy.csv', index=False)
    predictions = {name: pd.concat(parts, ignore_index=True) for name, parts in evaluation_wavs.items()}
    if MODE == 'full':
        for frame in predictions.values():
            if set(frame['file_path']) != set(USABLE_WAVS['file_path']):
                raise ValueError('Outer predictions must cover every usable WAV exactly once')
    pooled = OUTPUT / 'pooled_outer'
    metrics = {name: evaluate_predictions(frame, pooled / name, require_all=MODE == 'full')
               for name, frame in predictions.items()}
    pair_predictions(predictions, pooled / 'paired_predictions.csv')
    return [*hive_rows, *comparison_rows(metrics, 'pooled_outer', 'test')]

### 학습 실행

In [ ]:
LOCAL_AUDIO_SOURCES = set(local_manifest['source_id'])
if MODE != 'audit':
    # Keras also uses Python's temporary directory during model save and reload.
    with tempfile.TemporaryDirectory(dir=OUTPUT, prefix='keras_tmp_') as scratch:
        previous_tempdir = tempfile.tempdir
        try:
            tempfile.tempdir = scratch
            write_comparison(run_experiment())
        finally:
            tempfile.tempdir = previous_tempdir
save_json(OUTPUT / 'completed.json', {'experiment': EXPERIMENT.upper(), 'architecture': 'GAP',
                                      'mode': MODE, 'epoch_limit': EPOCHS,
                                      'training_performed': MODE != 'audit'})
print(f'Completed {EXPERIMENT.upper()} · {MODE} · {OUTPUT}')

## 5. 결과 보기 · 학습 곡선

완료된 실행이 있으면 아래 **보고 셀 네 개만** 순서대로 실행해 그림을 만들 수 있습니다. 프로젝트 루트에서 실행하면 최근 완료 실행을 선택합니다. 특정 실행을 열려면 첫 보고 셀에서 `REPORT_RUN_DIR`에 실행 폴더를 지정하세요.

A의 내부 fold는 train·val을 함께 표시하고, 고정 에포크 최종 재학습은 train 곡선을 표시합니다.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import Image, display

# Set a path here to view a specific run, or leave it unset to use the latest completed run.
REPORT_RUN_DIR = globals().get('REPORT_RUN_DIR', None)
REPORT_EXPERIMENT = 'a'
report_root = Path(globals().get('ROOT', Path.cwd()))
active_run = globals().get('OUTPUT')
report_run = None
if REPORT_RUN_DIR is not None:
    report_run = Path(REPORT_RUN_DIR).expanduser().resolve()
elif active_run is not None and (active_run / 'completed.json').is_file():
    if globals().get('MODE') != 'audit':
        report_run = active_run
else:
    run_root = report_root / 'artifacts/experiments' / f'ex_{REPORT_EXPERIMENT}'
    completed = sorted(run_root.glob('*/completed.json'))
    if completed:
        report_run = completed[-1].parent

if report_run is None:
    print('보고서를 만들 완료 실행이 없습니다. REPORT_RUN_DIR에 실행 폴더를 지정할 수 있습니다.')
else:
    metadata = json.loads((report_run / 'completed.json').read_text(encoding='utf-8'))
    if (metadata['experiment'] != REPORT_EXPERIMENT.upper()
            or metadata['architecture'] != 'GAP' or not metadata['training_performed']):
        raise ValueError(f'Expected a completed GAP experiment {REPORT_EXPERIMENT.upper()}: {report_run}')
    report_dir = report_run / 'report'
    report_dir.mkdir(exist_ok=True)
    print(f'{metadata["experiment"]} · {metadata["mode"]} · {report_run}')

MODEL_LABELS = {'cnn': 'CNN', 'cnn_mlp': 'CNN + MLP', 'cnn_lstm': 'CNN + LSTM'}

def save_report_figure(fig, path):
    fig.savefig(path, dpi=135, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    display(Image(filename=str(path)))

In [ ]:
def plot_stage_learning(stage, histories):
    fig, axes = plt.subplots(2, len(histories), figsize=(4.3 * len(histories), 5.5),
                             sharex='col', squeeze=False, layout='constrained')
    has_validation = all('val_loss' in table for table in histories.values())
    subtitle = 'Train and validation' if has_validation else 'Fixed-epoch refit · train only'
    fig.suptitle(f'{stage}  |  {subtitle}', fontsize=13, fontweight='bold')
    for column, (model, table) in enumerate(histories.items()):
        epochs = table['epoch']
        best_epoch = int(epochs.iloc[table['val_loss'].to_numpy().argmin()]) if has_validation else None
        for row, metric in enumerate(('accuracy', 'loss')):
            ax = axes[row, column]
            ax.plot(epochs, table[metric], color='#205B96', marker='o', markersize=3, label='Train')
            if best_epoch is not None:
                ax.plot(epochs, table[f'val_{metric}'], color='#A84D26', marker='s',
                        markersize=3, linestyle='--', label='Validation')
                ax.axvline(best_epoch, color='#636B75', linestyle=':', linewidth=1,
                           label=f'Best val loss · epoch {best_epoch}')
            ax.grid(axis='y', alpha=0.25)
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
            if len(epochs) == 1:
                ax.set_xticks(epochs)
            if row == 0:
                ax.set(title=MODEL_LABELS[model], ylabel='Accuracy', ylim=(0, 1.03))
                ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
                ax.legend(fontsize=8, loc='best')
            else:
                ax.set(xlabel='Epoch', ylabel='Loss')
    return fig

if report_run is not None:
    histories_by_stage = {}
    for path in sorted(report_run.glob('**/training_history.csv')):
        if path.parent.name not in MODEL_LABELS:
            continue
        stage = path.parent.parent.relative_to(report_run).as_posix()
        histories_by_stage.setdefault(stage, {})[path.parent.name] = pd.read_csv(path)
    if not histories_by_stage:
        raise FileNotFoundError(f'No training histories in {report_run}')
    stages = sorted(histories_by_stage, key=lambda stage: (stage.split('/')[0], stage.endswith('/final'), stage))
    for stage in stages:
        histories = histories_by_stage[stage]
        filename = 'learning_' + stage.replace('/', '__') + '.png'
        save_report_figure(plot_stage_learning(stage, histories), report_dir / filename)

## 6. 테스트 결과 시각화

저장된 테스트 추론 결과로 WAV·원본 단위 정확도, macro F1, 4클래스 혼동행렬을 표시합니다. 외부 테스트 벌통별 결과와 전체 외부 예측 집계를 나누어 표시합니다.

In [ ]:
def plot_test_results(title, model_dirs):
    results = {model: json.loads((directory / 'metrics.json').read_text(encoding='utf-8'))
               for model, directory in model_dirs.items()}
    fig, axes = plt.subplots(2, len(results) + 1, figsize=(4.0 * (len(results) + 1), 7.2),
                             squeeze=False, layout='constrained')
    fig.suptitle(f'{title}  |  Test set\nConfusion matrices show counts and shade each true-class row by proportion',
                 fontsize=13, fontweight='bold')
    positions = np.arange(len(results))
    for row, (unit, label) in enumerate((('wav', 'WAV'), ('source', 'Source probability mean'))):
        ax = axes[row, 0]
        for offset, (metric, name, color) in zip((-0.18, 0.18), (
                ('accuracy', 'Accuracy', '#205B96'), ('macro_f1', 'Macro F1', '#A84D26'))):
            values = [scores[unit][metric] for scores in results.values()]
            bars = ax.bar(positions + offset, values, width=0.34, color=color, label=name)
            ax.bar_label(bars, labels=[f'{value:.1%}' for value in values], fontsize=8, padding=3)
        ax.set_xticks(positions, [MODEL_LABELS[model] for model in results])
        supports = {scores[unit]['support'] for scores in results.values()}
        if len(supports) != 1:
            raise ValueError(f'{title}: models must use the same test set')
        ax.set(title=f'{label} · n={next(iter(supports)):,}', ylim=(0, 1.15))
        ax.set_yticks(np.linspace(0, 1, 5))
        ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
        ax.set_axisbelow(True)
        ax.grid(axis='y', alpha=0.25)
        ax.legend(fontsize=8, loc='upper right')
        for column, (model, directory) in enumerate(model_dirs.items(), start=1):
            ax = axes[row, column]
            matrix = pd.read_csv(directory / f'{unit}_confusion_matrix.csv', index_col=0).to_numpy()
            support = matrix.sum(axis=1)
            if matrix.shape != (4, 4) or support.tolist() != results[model][unit]['class_support']:
                raise ValueError(f'Confusion matrix and class counts disagree: {directory}')
            shares = np.divide(matrix, support[:, None], where=support[:, None] > 0,
                               out=np.zeros((4, 4), dtype=float))
            ax.imshow(shares, cmap='Blues', vmin=0, vmax=1)
            for i, j in np.ndindex(matrix.shape):
                ax.text(j, i, f'{matrix[i, j]:,}', ha='center', va='center',
                        color='white' if shares[i, j] > 0.55 else '#172433')
            ax.set_xticks(range(4), range(4))
            ax.set_yticks(range(4), [str(i) if count else f'{i} (N/A)' for i, count in enumerate(support)])
            ax.set(title=MODEL_LABELS[model], xlabel='Predicted class', ylabel='True class')
    return fig

if report_run is not None:
    test_scopes = {}
    if REPORT_EXPERIMENT == 'a':
        test_scopes['pooled_outer'] = (report_run / 'pooled_outer', False)
        for path in sorted(report_run.glob('outer_hive_*/final')):
            test_scopes[path.parent.name] = (path, True)
    else:
        test_scopes['fixed'] = (report_run / 'fixed', True)
    test_models = ['cnn_mlp'] if REPORT_EXPERIMENT == 'b' else ['cnn_mlp', 'cnn_lstm']
    for scope, (directory, split_subdir) in test_scopes.items():
        model_dirs = {name: directory / name / ('test' if split_subdir else '') for name in test_models}
        title = f'Experiment {REPORT_EXPERIMENT.upper()} · {scope.replace("_", " ")}'
        save_report_figure(plot_test_results(title, model_dirs), report_dir / f'test_{scope}.png')

## 7. 그림과 상세 결과 저장

그림은 실행 폴더의 `report/`에 저장됩니다. 아래 셀은 에포크 이력과 테스트 요약을 CSV로 내보내고, 학습 중 저장한 모델과 전처리 설정, 그림, 상세 지표, 예측 파일의 경로를 `artifact_index.csv`에 정리합니다.

In [ ]:
if report_run is not None:
    epoch_tables = [
        history.assign(stage=stage, model=model)
        for stage, histories in histories_by_stage.items() for model, history in histories.items()
    ]
    pd.concat(epoch_tables, ignore_index=True).to_csv(report_dir / 'epoch_history.csv', index=False, na_rep='N/A')
    summary_file = report_run / 'test_comparison.csv'
    if not summary_file.is_file():
        summary_file = report_run / 'comparison.csv'
    summary = pd.read_csv(summary_file, keep_default_na=False)
    test_summary = summary[summary['split'].eq('test')]
    if test_summary.empty:
        raise ValueError(f'No test rows in {summary_file}')
    test_summary.to_csv(report_dir / 'test_summary.csv', index=False)

    suffixes = {'.keras': 'model', '.png': 'figure', '.csv': 'details', '.json': 'details'}
    index_path = report_dir / 'artifact_index.csv'
    artifacts = [{'kind': suffixes[path.suffix], 'path': path.relative_to(report_run).as_posix(),
                  'bytes': path.stat().st_size}
                 for path in sorted(report_run.rglob('*'))
                 if path.suffix in suffixes and path != index_path and path.is_file()]
    pd.DataFrame(artifacts).to_csv(index_path, index=False)
    print(f'Models, figures and details | {index_path}')
    display(test_summary[['scope', 'hive_id', 'model', 'unit', 'accuracy', 'macro_f1',
                          'wavs', 'sources', *[f'recall_{i}' for i in range(4)]]])
    hive_accuracy = report_run / 'equal_hive_accuracy.csv'
    if hive_accuracy.is_file():
        print(f'Equal-weight mean accuracy by hive | {hive_accuracy}')
        display(pd.read_csv(hive_accuracy))